In [1]:
import os
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"  # Suppress Pygame support prompt
import pygame, sys
from tqdm import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt
import random
import tkinter as tk
from tkinter import simpledialog

from utils import (
    place_O, place_X, check_win, check_win_state, get_empty_spots,
    print_q_value, print_state_q_values, new_boards,
)
from render import (
    render_board, add_XO, check_win_update, prompt_for_vi_params,
    prompt_for_player_choice, prompt_for_eval_games, init_window,
    new_graphical_board, draw_background, draw_status_bar,
    draw_stats_panel, plot_win_rate,
)


In [2]:
board, logical_board = new_boards()
graphical_board = new_graphical_board()

to_move = 'X'


In [3]:
# Generate every reachable (board, player) state

def generate_all_states(board, player, states):
    board = tuple(tuple(row) for row in board)
    state = (board, player)

    # Add the current state
    states.add(state)

    # Stop expanding terminal states
    winner = check_win_state(board)
    if winner is not None:
        return

    # Generate every legal next move
    for i in range(3):
        for j in range(3):

            if board[i][j] == 0:

                new_board = [list(row) for row in board]
                new_board[i][j] = player

                next_player = 1 if player == 2 else 2

                generate_all_states(
                    new_board,
                    next_player,
                    states
                )


# Generate state space

states = set()

initial_board = [
    [0, 0, 0],
    [0, 0, 0],
    [0, 0, 0]
]

# Suppose player 1 starts
generate_all_states(initial_board, 1, states)

print(f"Total reachable states: {len(states)}")


# Initialize V(s) and Q(s,a)

state_values = {}
q_values = {}

for state in states:

    board, player = state

    # Terminal states have no actions
    if check_win_state(board) is not None:
        state_values[state] = 0.0
        q_values[state] = {}
        continue

    # Non-terminal state
    actions = get_empty_spots(board)

    state_values[state] = 0.0
    q_values[state] = {
        action: 0.0
        for action in actions
    }


# Count state-action pairs

count = 0

for state, actions in q_values.items():
    count += len(actions)

print(f"Total state-action pairs: {count}")

Total reachable states: 5478
Total state-action pairs: 16167


In [ ]:
# | Issue                                    | Importance                                              |
# | ---------------------------------------- | ------------------------------------------------------- |
# | X should minimize, O should maximize     | **Critical**                                            |
# | Evaluation should use a fixed policy     | **Critical**                                            |
# | `delta_v` not reset per evaluation sweep | **Bug**                                                 |
# | `delta_v < 0.8`                          | **Major**                                               |
# | No explicit policy representation        | **Conceptual/design issue**                             |
# | In-place value updates                   | Fine                                                    |
# | Fixed 30 evaluation iterations           | Fine, if intentionally using truncated policy iteration |


In [8]:
## policy iteration, loop through all states evaluate policy until state value convergence then

iterations, discount = prompt_for_vi_params()
player = 2
# inner iteration is fixed
inner_iteration = 30
delta_v = 0

## Generate initial policy
policy = {}
for state in state_values:
    board, player = state
    
    if check_win_state(board) is not None:
        # skip finished games
        continue
    
    # pick the first possible action
    policy[state] = get_empty_spots(board)[0]
    

## loop 
for _ in tqdm(range(iterations), desc="Value Iteration", ncols=80): 
    ## Policy evaluation
    for _ in range(inner_iteration):
        for state in state_values:
            board, player = state
                    
            # Terminal state
            if check_win_state(board) is not None:
                continue
            
            initial_value = state_values[state]
            
            # update state_value based on initial policy
            action = policy[state]
            
            new_board = [list(row) for row in board]
            new_board[action[0]][action[1]] = player  
            new_state = tuple(tuple(row) for row in new_board)
              
            winner = check_win_state(new_board)
            if winner == 1:
                reward = -1.0
            elif winner == 2:
                reward = 1.0
            elif winner == 0:
                reward = 0.0
            else:
                reward = -0.05     
                
            if winner is not None:
                future_value = 0.0
            
            else:
                # Switch to the other player
                next_player = 1 if player == 2 else 2

                next_state = (new_state, next_player)
                
                future_value = state_values[next_state]
                
            state_values[state] =  reward + discount * future_value
            
            delta_v =  max(delta_v, abs(initial_value - state_values[state]))
        if delta_v < 0.05:
            break
        delta_v = 0
    
    ## Policy improvement
    for state in state_values:
        board, player = state
        # Terminal state
        if check_win_state(board) is not None:
            continue
        
        for action in get_empty_spots(board):
            
            new_board = [list(row) for row in board]
            new_board[action[0]][action[1]] = player  
            new_state = tuple(tuple(row) for row in new_board)
              
            winner = check_win_state(new_board)
            if winner == 1:
                reward = -1.0
            elif winner == 2:
                reward = 1.0
            elif winner == 0:
                reward = 0.0
            else:
                reward = -0.05     
                
            if winner is not None:
                future_value = 0.0
            
            else:
                # Switch to the other player
                next_player = 1 if player == 2 else 2

                next_state = (new_state, next_player)
                
                future_value = state_values[next_state]
                
            q_values[state][action] = reward + discount * future_value
        
        if player == 2:
            action, _ = max(q_values.get(state, {}).items(), key=lambda x: x[1])
        else:
            action, _ = min(q_values.get(state, {}).items(), key=lambda x: x[1])
        policy[state] = action
               

Value Iteration: 100%|████████████████████████| 100/100 [00:25<00:00,  3.97it/s]


In [9]:
## quick test

check_state = (((1, 2, 2), (1, 1, 2), (0, 1, 0)),2)
print(max(q_values.get(check_state, {}).items(), key=lambda x: x[1]))


((2, 2), 1.0)


In [10]:
## random play

to_move = "X"

game_finished = False

count = 0
win_count = 0
loss_count = 0
stalemate_count = 0
epsilon_greedy = 1.0

win_rate_history = []
game_intervals = []

max_episodes = prompt_for_eval_games()
player_choice = prompt_for_player_choice()

pbar = tqdm(total=max_episodes, desc="Training", ncols=80)

game_finished = True

while count < max_episodes:
    # (1) The user is about to place X if it's X's turn
    # If game is finished, do resets
    if game_finished:
        board, logical_board = new_boards()
        graphical_board = new_graphical_board()
        to_move = player_choice
        
        game_finished = False

    if to_move == "X":
        empty_spots = get_empty_spots(logical_board)
        row, col = random.choice(empty_spots)
    
        place_X(board, logical_board, (row, col))
        
        to_move = 'O'

    else:
        last_state = tuple(tuple(row) for row in logical_board)

        # pick action using the max q_value
        action = max(q_values.get((last_state,2), {}).items(), key=lambda x: x[1])
        row, col = action[0]

        place_O(board, logical_board, (row,col))

        new_state = tuple(tuple(row) for row in logical_board)
        to_move = 'X'
    
    winner = check_win(board)
    if winner is not None:
        if winner == "X":
            loss_count += 1
        elif winner == "O":
            win_count += 1
        else:
            stalemate_count += 1

        game_finished = True
        count += 1
        pbar.update(1) 

pbar.close()

print(f'=========== Training Results ===========')
print(f'At {count} games, the current stats are:')
print(f'Wins: {win_count}')
print(f'Losses: {loss_count}')
print(f'Stalemate: {stalemate_count}')
print(f'Current epsilon value: {epsilon_greedy}')
print(f'Win rate is {(win_count/count) * 100}%')

Training: 100%|█████████████████████████████| 100/100 [00:00<00:00, 8293.24it/s]

=========== Training Results ===========
At 100 games, the current stats are:
Wins: 71
Losses: 0
Stalemate: 29
Current epsilon value: 1.0
Win rate is 71.0%


In [ ]:
# print_q_value(q_values)

play_count = 0
play_win_count = 0
play_loss_count = 0
play_stalemate_count = 0
epsilon_greedy = 0.5

win_rate_history = []
game_intervals = []

game_finished = True

SCREEN, BOARD, X_IMG, O_IMG, FONT, SMALL_FONT = init_window()

draw_background(SCREEN, BOARD)

pygame.display.update()

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            if play_count != 0:
                print(f'At {play_count} games, the current stats are:')
                print(f'Wins: {play_win_count}')
                print(f'Losses: {play_loss_count}')
                print(f'Stalemate: {play_stalemate_count}')
                print(f'Win rate is {(play_win_count/play_count) * 100}%')
                # print_q_value(q_values)

                plot_win_rate(game_intervals, win_rate_history)
            else:
                print("You have not played yet!")
            pygame.quit()
            sys.exit()    
        
        if event.type == pygame.MOUSEBUTTONDOWN:
            # (1) The user is about to place X if it's X's turn
            # If game is finished, do resets
            if game_finished:
                board, logical_board = new_boards()
                graphical_board = new_graphical_board()
                to_move = player_choice
                
                draw_background(SCREEN, BOARD)
                game_finished = False
                pygame.display.update()

            if to_move == "X":

                add_XO(board, graphical_board, "X", logical_board, SCREEN, X_IMG, O_IMG)

                render_board(board, X_IMG, O_IMG, graphical_board)
                for i in range(3):
                    for j in range(3):
                        if graphical_board[i][j][0] is not None:
                            SCREEN.blit(graphical_board[i][j][0], graphical_board[i][j][1])
                draw_status_bar(SCREEN, SMALL_FONT, "Agent turn")
                draw_stats_panel(SCREEN, SMALL_FONT, play_win_count, play_loss_count, play_stalemate_count, label="Value Iteration")
                pygame.display.update()

                to_move = "O"
            # The reason there has to be an else is that it should check after every move if there is a winner
            # For example, if X (you) moves last and you win, O will still go despite the game being over and then O will win 
            # Since the terminal states are checked after O in the code, even though your move (X) should've ended the game
            else:
                last_state = tuple(tuple(row) for row in logical_board)
                
                # pick action using the max q_value
                action = max(q_values.get((last_state,2), {}).items(), key=lambda x: x[1])
                row, col = action[0]
        
                place_O(board, logical_board, (row,col))

                render_board(board, X_IMG, O_IMG, graphical_board)
                for i in range(3):
                    for j in range(3):
                        if graphical_board[i][j][0] is not None:
                            SCREEN.blit(graphical_board[i][j][0], graphical_board[i][j][1])
                draw_status_bar(SCREEN, SMALL_FONT, "Your turn")
                draw_stats_panel(SCREEN, SMALL_FONT, play_win_count, play_loss_count, play_stalemate_count, label="Value Iteration")
                pygame.display.update()

                new_state = tuple(tuple(row) for row in logical_board)

                to_move = 'X'

            winner = check_win_update(board, graphical_board, SCREEN)
            if winner is not None:
                if winner == "X":
                    reward = -1
                    play_loss_count += 1
                elif winner == "O":
                    reward = 1
                    play_win_count += 1
                else:
                    reward = 0
                    play_stalemate_count += 1

                game_finished = True
                play_count += 1
                win_rate = play_win_count / play_count 
                win_rate_history.append(win_rate)  
                game_intervals.append(play_count) 
                if play_count % 3 == 0: 
                    print(f'At {play_count} games, the current stats are:')
                    print(f'Wins: {play_win_count}')
                    print(f'Losses: {play_loss_count}')
                    print(f'Stalemate: {play_stalemate_count}')
                    print(f'Win rate is {win_rate * 100}%')


ValueError: max() arg is an empty sequence

: 